# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,SPY,14,10,0.139520,2026-06-15 18:17:48+00:00
1,SPCE,17,4,-0.086425,2026-06-13 03:19:03+00:00
2,RKLB,5,4,0.330425,2026-06-14 16:13:15+00:00
3,NBIS,4,2,0.672600,2026-06-12 00:28:27+00:00
4,SMCI,3,2,-0.421600,2026-06-10 20:18:16+00:00
5,ALAB,3,2,0.672600,2026-06-12 00:28:27+00:00
6,CRWV,3,2,0.672600,2026-06-12 00:28:27+00:00
7,TER,3,2,0.672600,2026-06-12 00:28:27+00:00
8,AVGO,3,2,0.180600,2026-06-15 22:51:44+00:00
9,LUNR,2,2,-0.011750,2026-06-14 16:13:15+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['SPY', 'SPCE', 'RKLB', 'NBIS']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
def safe_latest_value(in_df, column_name, caster=float):
    if in_df.empty or column_name not in in_df.columns:
        return None
    col_data = in_df[column_name].dropna()
    if col_data.empty:
        return None
    return caster(col_data.iloc[-1])


price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": safe_latest_value(features_df, "Close", float),
        "latest_rsi": safe_latest_value(features_df, "RSI", float),
        "latest_macd": safe_latest_value(features_df, "MACD", float),
        "return_21d": safe_latest_value(features_df, "21D Return", float),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,NBIS,251,283.609985,68.285047,21.552057,0.478521,8.728554,C:\Users\big_j\PycharmProjects\Project2-Sentim...,C:\Users\big_j\PycharmProjects\Project2-Sentim...,C:\Users\big_j\PycharmProjects\Project2-Sentim...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== SPY latest metrics =====
                 Close    Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return
Date                                                                                                                                     
2026-06-15  754.830017  60176400           10.577219          742.302999           713.325064  60.394596  4.520982   0.021117    0.008902
2026-06-16  750.330017  67093100            9.620226          743.380332           714.173011  57.047702  4.697111   0.018018    0.015098
2026-06-17  740.960022  85945200            8.896643          743.953333           714.786775  50.742135  4.034112   0.021408    0.003127
2026-06-18  746.739990  80875700            8.700150          744.383665           715.518857  54.111591  3.929775   0.012172    0.017731
2026-06-22  744.390015  45157614            8.357698          744.810665           716.180279  52.538041  3.615784   0.003559    0.004236

=

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\big_j\AppData\Local\Temp\ipykernel_219560\3132690536.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== SPY regression results =====
                 Close  Predicted Close
Date                                   
2026-06-08  739.219971       752.837338
2026-06-09  737.049988       748.469607
2026-06-10  725.429993       739.387080
2026-06-11  737.760010       746.293634
2026-06-12  741.750000       752.243315
2026-06-15  754.830017       760.840587
2026-06-16  750.330017       760.294239
2026-06-17  740.960022       755.916985
2026-06-18  746.739990       757.953428
2026-06-22  744.390015       755.437585

===== SPCE regression results =====
            Close  Predicted Close
Date                              
2026-06-08   4.12         3.520206
2026-06-09   4.59         3.682944
2026-06-10   4.71         3.907719
2026-06-11   5.73         3.869782
2026-06-12   3.91         3.119943
2026-06-15   3.56         3.473807
2026-06-16   3.35         3.344302
2026-06-17   3.49         3.350273
2026-06-18   3.56         3.217399
2026-06-22   3.19         3.225706

===== RKLB regression resu

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "SPY",
    "SPCE",
    "RKLB",
    "NBIS"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\outputs\\latest_wsb_analysis\\financial_metrics_summary.csv",
  "output_dir": "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\outputs\\latest_wsb_analysis"
}
